In [1]:

from soundmining_library.piece import Piece

piece = Piece()
piece.start(should_send_to_score=False)

In [2]:
import random
from dataclasses import dataclass
from enum import StrEnum
from itertools import permutations

import ipywidgets as widgets
from soundmining_library.generative import MarkovChain, pan_point, random_int_range, random_range
from soundmining_library.modular.instrument import AddAction, AudioInstrument
from soundmining_library.modular_v2.synth_player_v2 import SynthPlayerV2
from soundmining_library.spectrum import make_fact, make_spectrum, make_undertone_spectrum
from soundmining_library.supercollider_client import SupercolliderClient
from soundmining_library.supercollider_receiver import ExtendedNoteHandler, PatchArguments

static_control = piece.instruments.static_control
sine_control = piece.instruments.sine_control
line_control = piece.instruments.line_control
two_block_sine_control = piece.instruments.two_block_sine_control
impulse_osc = piece.instruments.impulse_osc
dust_osc = piece.instruments.dust_osc
signal_sum = piece.instruments.signal_sum
signal_multiply = piece.instruments.signal_multiply
lf_noise_osc = piece.instruments.lf_noise_osc

SOUNDPATH = piece.environment.sound_path

Sounds = StrEnum("Sounds", ["H1", "H2", "K1", "Ka1", "Ki1", "Ko1", "S1", "S2"])


@dataclass
class SoundData:
    sound: Sounds
    file_name: str
    duration: float
    fundamental: float
    first_partial: float
    start_time: float
    end_time: float
    peak_time: float

    def make_fact(self) -> float:
        return make_fact(fundamental=self.fundamental, first_partial=self.first_partial)

    def make_spectrum(self) -> list[float]:
        fact = self.make_fact()
        return make_spectrum(fundamental=self.fundamental, fact=fact, size=SPECTRUM_SIZE)

    def make_undertone_spectrum(self) -> list[float]:
        fact = self.make_fact()
        return make_undertone_spectrum(fundamental=self.fundamental, fact=fact, size=SPECTRUM_SIZE)

    def get_relative_peak_time(self, reverse: bool = False) -> tuple[float, float]:
        relative_peak_time = (self.peak_time - self.start_time) / (self.end_time - self.start_time)
        if reverse:
            relative_peak_time = 1.0 - relative_peak_time
        relative_times = (relative_peak_time, 1.0 - relative_peak_time)

        return relative_times

    def get_relative_start_end(self) -> tuple[float, float]:
        return (self.start_time / self.duration, self.end_time / self.duration)

    def make_rate(self, second_freq: float) -> float:
        return second_freq / self.fundamental

    def add_sound_to_synth_player(self, synth_player: SynthPlayerV2):
        synth_player.add_sound(self.sound, self.file_name, self.start_time, self.end_time)

    def get_play_duration(self, rate: float) -> float:
        return (self.end_time - self.start_time) * rate


all_sound_data = {
    Sounds.H1: SoundData(
        sound=Sounds.H1,
        file_name=f"{SOUNDPATH}/H1.aif",
        duration=1.781,
        fundamental=959,
        first_partial=1565,
        start_time=0.234,
        end_time=1.395,
        peak_time=0.410,
    ),
    Sounds.H2: SoundData(
        sound=Sounds.H2,
        file_name=f"{SOUNDPATH}/H2.aif",
        duration=1.414,
        fundamental=933,
        first_partial=1307,
        start_time=0.105,
        end_time=1.216,
        peak_time=0.281,
    ),
    Sounds.K1: SoundData(
        sound=Sounds.K1,
        file_name=f"{SOUNDPATH}/K1.aif",
        duration=0.896,
        fundamental=490,
        first_partial=1696,
        start_time=0.196,
        end_time=0.779,
        peak_time=0.233,
    ),
    Sounds.Ka1: SoundData(
        sound=Sounds.Ka1,
        file_name=f"{SOUNDPATH}/Ka1.aif",
        duration=0.757,
        fundamental=1350,
        first_partial=1678,
        start_time=0.249,
        end_time=0.714,
        peak_time=0.315,
    ),
    Sounds.Ki1: SoundData(
        sound=Sounds.Ki1,
        file_name=f"{SOUNDPATH}/Ki1.aif",
        duration=0.649,
        fundamental=492,
        first_partial=636,
        start_time=0.164,
        end_time=0.570,
        peak_time=0.211,
    ),
    Sounds.Ko1: SoundData(
        sound=Sounds.Ko1,
        file_name=f"{SOUNDPATH}/Ko1.aif",
        duration=0.721,
        fundamental=567,
        first_partial=1080,
        start_time=0.169,
        end_time=0.630,
        peak_time=0.220,
    ),
}

piece.reset()

for sound_data in all_sound_data.values():
    sound_data.add_sound_to_synth_player(piece.synth_player)

(
    piece.synth_player.add_sound(Sounds.S1, f"{SOUNDPATH}/S1.aif", 0.170, 1.834).add_sound(
        Sounds.S2, f"{SOUNDPATH}/S1.aif", 0.192, 1.693
    )
)
piece.synth_player.start()

peaks = {
    Sounds.H1: [380, 812, 959, 1128, 1426, 1568],
    Sounds.H2: [423, 564, 643, 809, 936, 1057, 1337, 1522],
    Sounds.K1: [420, 490, 1070, 1536, 1683, 1779],
    Sounds.Ka1: [564, 658, 1052, 1285, 1362, 1669, 1734, 1874, 1980, 2038, 2365],
    Sounds.Ki1: [156, 399, 492, 563, 636, 915, 1399, 1645, 1897, 2247, 2438, 2570, 2647],
    Sounds.Ko1: [448, 569, 602, 912, 1102, 1198],
}

freq_peak = {
    Sounds.H1: 1406,
    Sounds.H2: 1195,
    Sounds.K1: 422,
    Sounds.Ka1: 1617,
    Sounds.Ki1: 2906,
    Sounds.Ko1: 984,
    Sounds.S1: 8227,
    Sounds.S2: 5273,
}

time_peak = {
    Sounds.H1: 0.410,
    Sounds.H2: 0.281,
    Sounds.K1: 0.233,
    Sounds.Ka1: 0.315,
    Sounds.Ki1: 0.211,
    Sounds.Ko1: 0.220,
    Sounds.S1: 0.584,
    Sounds.S2: 0.690,
}

FilterType = StrEnum("FilterType", ["LOW", "MIDDLE_LOW", "MIDDLE_HIGH", "HIGH"])
NoteType = StrEnum("NoteType", ["LOW", "MIDDLE_LOW", "MIDDLE_HIGH", "HIGH"])


SPECTRUM_SIZE = 50

Duration = StrEnum("Duration", ["SHORT", "MEDIUM", "LONG"])

TimeStart = StrEnum("TimeStart", ["EARLY", "MIDDLE", "LATE"])
# rate_note = 4

# status = widgets.Output()
# is_grain_pos_reverse = widgets.Checkbox(value=False, description='Grain Pos Reverse')

# box = widgets.VBox([is_grain_pos_reverse, status])
# display(box)


class MyHandler(ExtendedNoteHandler):
    def __init__(self, client: SupercolliderClient):
        self.rate_note = 4
        super().__init__(client)

    def handle_ring_modulate_note(self, patch_arguments: PatchArguments):
        match patch_arguments.note:
            case 0:
                sound = Sounds.H1
            case 1:
                sound = Sounds.H2
            case 2:
                sound = Sounds.K1
            case 3:
                sound = Sounds.Ka1
            case 4:
                sound = Sounds.Ki1
            case 5:
                sound = Sounds.Ko1
            case _:
                return  # Unknown note, do nothing
        match patch_arguments.octave:
            case 2:
                ring_pitch = peaks[sound][0]
            case 3:
                ring_pitch = peaks[sound][1]
            case 4:
                ring_pitch = peaks[sound][-1]
            case _:
                return  # Unknown octave, do nothing

        amp = patch_arguments.amp * 3
        sound_duration = piece.synth_player.get_sound(sound).duration(1.0)
        osc = piece.synth_player.note().sound_mono(sound, 1.0, static_control(amp)).audio_stack.pop()
        mod = (
            piece.synth_player.note()
            .sine(freq=static_control(ring_pitch), amp=static_control(amp * 0.5))
            .audio_stack.pop()
        )
        ring = signal_multiply(osc, mod).add_action(AddAction.TAIL_ACTION)
        (
            piece.synth_player.note()
            .push(ring)
            .pan(static_control(0))
            .play(start_time=patch_arguments.start, duration=sound_duration)
        )

    def filter_range(self, filter_type: FilterType) -> tuple[float | None, float | None]:
        match filter_type:
            case FilterType.LOW:
                return None, random_range(100, 300)
            case FilterType.MIDDLE_LOW:
                return random_range(100, 300), random_range(800, 1200)
            case FilterType.MIDDLE_HIGH:
                return random_range(800, 1200), random_range(2500, 3500)
            case FilterType.HIGH:
                return random_range(2500, 3500), None
            case _:
                return None, None

    def handle_filter_range_note(self, patch_arguments: PatchArguments):
        match patch_arguments.note:
            case 0:
                sound = Sounds.H1
            case 1:
                sound = Sounds.H2
            case 2:
                sound = Sounds.K1
            case 3:
                sound = Sounds.Ka1
            case 4:
                sound = Sounds.Ki1
            case 5:
                sound = Sounds.Ko1
            case _:
                return  # Unknown note, do nothing
        match patch_arguments.octave:
            case 2:
                filter_type = FilterType.LOW
            case 3:
                filter_type = FilterType.MIDDLE_LOW
            case 4:
                filter_type = FilterType.MIDDLE_HIGH
            case 5:
                filter_type = FilterType.HIGH
            case _:
                return  # Unknown octave, do nothing

        low_filter, high_filter = self.filter_range(filter_type)
        rate = random_range(0.95, 1.05)
        pan = random_range(-0.99, 0.99)

        sound = (
            piece.synth_player.note().sound_mono(sound, rate, static_control(1.0))
            # .sound_mono(sound, rate, two_block_sine_control((0, 1.0, 0), (0.5, 0.5)))
        )
        if low_filter is not None:
            sound = sound.mono_high_pass_filter(static_control(low_filter))
        if high_filter is not None:
            sound = sound.mono_low_pass_filter(static_control(high_filter))
        (sound.pan(static_control(pan)).play(start_time=patch_arguments.start))

    def handle_reverse_note(self, patch_arguments: PatchArguments):
        match patch_arguments.note:
            case 0:
                sound = Sounds.H1
            case 1:
                sound = Sounds.H2
            case 2:
                sound = Sounds.K1
            case 3:
                sound = Sounds.Ka1
            case 4:
                sound = Sounds.Ki1
            case 5:
                sound = Sounds.Ko1
            case _:
                return  # Unknown note, do nothing

        rate = random_range(0.95, 1.05)
        pan = random_range(-0.99, 0.99)
        match patch_arguments.octave:
            case 2:
                start = piece.synth_player.get_sound(sound).end
                end = piece.synth_player.get_sound(sound).start
            case 3:
                start = piece.synth_player.get_sound(sound).start
                end = piece.synth_player.get_sound(sound).end
            case _:
                return  # Unknown octave, do nothing

        (
            piece.synth_player.note()
            .sound_mono(sound, rate, static_control(1.0), start_override=start, end_override=end)
            .pan(static_control(pan))
            .play(start_time=patch_arguments.start)
        )

    def get_relative_peak_time(self, sound: Sounds, reverse: bool = False) -> tuple[float, float]:
        start_time = piece.synth_player.get_sound(sound).start
        end_time = piece.synth_player.get_sound(sound).end
        peak_time = time_peak[sound]
        relative_peak_time = (peak_time - start_time) / (end_time - start_time)
        if reverse:
            relative_peak_time = 1.0 - relative_peak_time
        relative_times = (relative_peak_time, 1.0 - relative_peak_time)

        return relative_times

    def handle_sine_volume_note(self, patch_arguments: PatchArguments):
        match patch_arguments.note:
            case 0:
                sound = Sounds.H1
            case 1:
                sound = Sounds.H2
            case 2:
                sound = Sounds.K1
            case 3:
                sound = Sounds.Ka1
            case 4:
                sound = Sounds.Ki1
            case 5:
                sound = Sounds.Ko1
            case _:
                return

        rate = random_range(0.95, 1.05)
        pan = random_range(-0.99, 0.99)
        match patch_arguments.octave:
            case 2:
                peak_start, peak_end = self.get_relative_peak_time(sound)
            case 3:
                peak_start, peak_end = self.get_relative_peak_time(sound, reverse=True)
            case 4:
                peak_start, peak_end = (0.5, 0.5)
            case _:
                return

        print(f"peak_start: {peak_start}, peak_end: {peak_end}")

        (
            piece.synth_player.note()
            .sound_mono(sound, rate, two_block_sine_control((0, 1.0, 0), (peak_start, peak_end)))
            .pan(static_control(pan))
            .play(start_time=patch_arguments.start)
        )

    def get_grain_duration(self, duration: Duration) -> float:
        match duration:
            case Duration.SHORT:
                return random_range(0.01, 0.03)
            case Duration.MEDIUM:
                return random_range(0.1, 0.3)
            case Duration.LONG:
                return random_range(1.5, 1.9)
            case _:
                return 0.1

    rate_note = 4

    status = widgets.Output()
    is_grain_pos_reverse = widgets.Checkbox(value=False, description="Grain Pos Reverse")

    sound_select = widgets.RadioButtons(
        options=[Sounds.H1, Sounds.H2, Sounds.K1, Sounds.Ka1, Sounds.Ki1, Sounds.Ko1],
        value=Sounds.H1,
        description="Grain Sound",
        disabled=False,
    )

    box = widgets.VBox([is_grain_pos_reverse, sound_select, status])
    # display(box)
    # display(is_grain_pos_reverse)
    # display(status)

    def handle_grain_buf_with_spectrum(self, patch_arguments: PatchArguments):
        match patch_arguments.octave:
            case 2:
                self.rate_note = patch_arguments.note
            case 3:
                sound_data = all_sound_data[self.sound_select.value]
                # sound_data = all_sound_data[Sounds.Ki1]
                sound = sound_data.sound
                pan = random_range(-0.99, 0.99)

                spectrum = sound_data.make_undertone_spectrum()
                freq = spectrum[patch_arguments.note]
                freq_control = static_control(freq)
                rate = sound_data.make_rate(spectrum[self.rate_note])

                grain_trigger_bus = impulse_osc(static_control(1.0), freq_control).add_action(AddAction.TAIL_ACTION)
                grain_duration_bus = static_control(self.get_grain_duration(Duration.SHORT))
                grain_rate_bus = static_control(rate)

                grain_pos_reverse = self.is_grain_pos_reverse.value
                relative_start, relative_end = sound_data.get_relative_start_end()

                pos_start = relative_start * random_range(1.0, 1.2)
                pos_end = relative_end * random_range(0.8, 0.95)
                if grain_pos_reverse:
                    pos_start, pos_end = pos_end, pos_start

                grain_pos_bus = line_control(pos_start, pos_end)
                duration = random_range(5, 8)

                self.status.outputs = []
                self.status.append_stdout(
                    f"freq {freq}, rate {rate} pos_start {pos_start}, pos_end {pos_end} sound {sound}\n"
                )

                (
                    piece.synth_player.note()
                    .mono_grain_buf(sound, grain_trigger_bus, grain_duration_bus, grain_rate_bus, grain_pos_bus)
                    .mono_volume(sine_control(0, 4.0))
                    .pan(static_control(pan))
                    .play(start_time=patch_arguments.start, duration=duration)
                )
            case 4:
                match patch_arguments.note:
                    case 0:
                        sound = Sounds.H1
                    case 1:
                        sound = Sounds.H2
                    case 2:
                        sound = Sounds.K1
                    case 3:
                        sound = Sounds.Ka1
                    case 4:
                        sound = Sounds.Ki1
                    case 5:
                        sound = Sounds.Ko1
                    case _:
                        return  # Unknown note, do nothing

                rate = random_range(0.95, 1.05)
                pan = random_range(-0.99, 0.99)
                (
                    piece.synth_player.note()
                    .sound_mono(sound, rate, static_control(1.0))
                    .pan(static_control(pan))
                    .play(start_time=patch_arguments.start)
                )

            case _:
                return  # Unknown octave, do nothing

    def handle_note(self, patch_arguments: PatchArguments):
        self.handle_grain_buf_with_spectrum(patch_arguments)


def filter_range(filter_type: FilterType) -> tuple[float | None, float | None]:
    match filter_type:
        case FilterType.LOW:
            return None, random_range(100, 300)
        case FilterType.MIDDLE_LOW:
            return random_range(100, 300), random_range(800, 1200)
        case FilterType.MIDDLE_HIGH:
            return random_range(800, 1200), random_range(2500, 3500)
        case FilterType.HIGH:
            return random_range(2500, 3500), None
        case _:
            return None, None


class MyHandler2(ExtendedNoteHandler):
    def __init__(self, client: SupercolliderClient):
        super().__init__(client)
        self._rate_note = 4

    def play_filtered_sound(
        self,
        start_time: float,
        rate: float,
        sound: Sounds,
        filter_type: FilterType,
        sound_start_time: float,
        sound_end_time: float,
    ):
        match filter_type:
            case FilterType.LOW:
                pan = pan_point([(-0.25, 0.25)])
                volume = random_range(0.35, 0.99) * 6
            case FilterType.MIDDLE_LOW:
                pan = pan = pan_point([(-0.50, -0.25), (0.25, 0.50)])
                volume = random_range(0.35, 0.99) * 4
            case FilterType.MIDDLE_HIGH:
                pan = pan_point([(-0.75, -0.50), (0.50, 0.75)])
                volume = random_range(0.35, 0.99) * 3
            case FilterType.HIGH:
                pan = pan_point([(-0.99, -0.75), (0.75, 0.99)])
                volume = random_range(0.35, 0.99) * 2

        low_filter, high_filter = filter_range(filter_type)
        the_sound = piece.synth_player.note().sound_mono(
            sound, rate, static_control(volume), start_override=sound_start_time, end_override=sound_end_time
        )
        if low_filter is not None:
            the_sound = the_sound.mono_high_pass_filter(static_control(low_filter))
        if high_filter is not None:
            the_sound = the_sound.mono_low_pass_filter(static_control(high_filter))
        (the_sound.pan(static_control(pan)).play(start_time=start_time))

    def handle_filtered_notes(self, patch_arguments: PatchArguments):
        # reverse = random.choice([True, False])
        reverse = False
        match patch_arguments.note:
            case 0:
                sound = Sounds.H1
            case 1:
                sound = Sounds.H2
            case 2:
                sound = Sounds.K1
            case 3:
                sound = Sounds.Ka1
            case 4:
                sound = Sounds.Ki1
            case 5:
                sound = Sounds.Ko1
            case _:
                return  # Unknown note, do nothing

        sound_data = all_sound_data[sound]
        match patch_arguments.octave:
            case 2:
                filter_type = FilterType.LOW
            case 3:
                filter_type = FilterType.MIDDLE_LOW
            case 4:
                filter_type = FilterType.MIDDLE_HIGH
            case 5:
                filter_type = FilterType.HIGH
            case _:
                return  # Unknown octave, do nothing

        sound_start_time, sound_end_time = sound_data.start_time, sound_data.end_time
        if reverse:
            sound_start_time, sound_end_time = sound_end_time, sound_start_time

        rate = random_range(0.95, 1.05)

        self.play_filtered_sound(
            start_time=patch_arguments.start,
            rate=rate,
            sound=sound,
            filter_type=filter_type,
            sound_start_time=sound_start_time,
            sound_end_time=sound_end_time,
        )

    long_sound_chain = MarkovChain(
        {
            (Sounds.H1, Sounds.H1): {
                (Sounds.H1, Sounds.H1): 0.33,
                (Sounds.H2, Sounds.H2): 0.33,
                (Sounds.H1, Sounds.H2): 0.33,
            },
            (Sounds.H2, Sounds.H2): {
                (Sounds.H1, Sounds.H1): 0.33,
                (Sounds.H2, Sounds.H2): 0.33,
                (Sounds.H1, Sounds.H2): 0.33,
            },
            (Sounds.H1, Sounds.H2): {
                (Sounds.H1, Sounds.H1): 0.33,
                (Sounds.H2, Sounds.H2): 0.33,
                (Sounds.H1, Sounds.H2): 0.33,
            },
        },
        (Sounds.H1, Sounds.H1),
    )

    low_filter_type_chain = MarkovChain(
        {
            FilterType.LOW: {FilterType.LOW: 0.4, FilterType.MIDDLE_LOW: 0.5, FilterType.MIDDLE_HIGH: 0.1},
            FilterType.MIDDLE_LOW: {FilterType.LOW: 0.5, FilterType.MIDDLE_LOW: 0.4, FilterType.MIDDLE_HIGH: 0.1},
            FilterType.MIDDLE_HIGH: {FilterType.LOW: 0.5, FilterType.MIDDLE_LOW: 0.5, FilterType.MIDDLE_HIGH: 0.0},
        },
        FilterType.LOW,
    )

    high_filter_type_chain = MarkovChain(
        {
            FilterType.HIGH: {FilterType.HIGH: 0.4, FilterType.MIDDLE_HIGH: 0.5, FilterType.MIDDLE_LOW: 0.1},
            FilterType.MIDDLE_HIGH: {FilterType.HIGH: 0.5, FilterType.MIDDLE_HIGH: 0.4, FilterType.MIDDLE_LOW: 0.1},
            FilterType.MIDDLE_LOW: {FilterType.HIGH: 0.5, FilterType.MIDDLE_HIGH: 0.5, FilterType.MIDDLE_LOW: 0.0},
        },
        FilterType.HIGH,
    )

    long_reverse_chain = MarkovChain({True: {True: 0.0, False: 1.0}, False: {True: 0.1, False: 0.9}}, False)

    short_sound_chain = MarkovChain(
        {
            (Sounds.Ka1, Sounds.Ka1): {
                (Sounds.Ka1, Sounds.Ka1): 0.0,
                (Sounds.Ka1, Sounds.Ki1): 0.1,
                (Sounds.Ka1, Sounds.Ko1): 0.1,
                (Sounds.Ki1, Sounds.Ka1): 0.1,
                (Sounds.Ki1, Sounds.Ki1): 0.1,
                (Sounds.Ki1, Sounds.Ko1): 0.1,
                (Sounds.Ko1, Sounds.Ka1): 0.1,
                (Sounds.Ko1, Sounds.Ki1): 0.1,
                (Sounds.Ko1, Sounds.Ko1): 0.1,
            },
            (Sounds.Ka1, Sounds.Ki1): {
                (Sounds.Ka1, Sounds.Ka1): 0.1,
                (Sounds.Ka1, Sounds.Ki1): 0.0,
                (Sounds.Ka1, Sounds.Ko1): 0.1,
                (Sounds.Ki1, Sounds.Ka1): 0.1,
                (Sounds.Ki1, Sounds.Ki1): 0.1,
                (Sounds.Ki1, Sounds.Ko1): 0.1,
                (Sounds.Ko1, Sounds.Ka1): 0.1,
                (Sounds.Ko1, Sounds.Ki1): 0.1,
                (Sounds.Ko1, Sounds.Ko1): 0.1,
            },
            (Sounds.Ka1, Sounds.Ko1): {
                (Sounds.Ka1, Sounds.Ka1): 0.1,
                (Sounds.Ka1, Sounds.Ki1): 0.1,
                (Sounds.Ka1, Sounds.Ko1): 0.0,
                (Sounds.Ki1, Sounds.Ka1): 0.1,
                (Sounds.Ki1, Sounds.Ki1): 0.1,
                (Sounds.Ki1, Sounds.Ko1): 0.1,
                (Sounds.Ko1, Sounds.Ka1): 0.1,
                (Sounds.Ko1, Sounds.Ki1): 0.1,
                (Sounds.Ko1, Sounds.Ko1): 0.1,
            },
            (Sounds.Ki1, Sounds.Ka1): {
                (Sounds.Ka1, Sounds.Ka1): 0.1,
                (Sounds.Ka1, Sounds.Ki1): 0.1,
                (Sounds.Ka1, Sounds.Ko1): 0.1,
                (Sounds.Ki1, Sounds.Ka1): 0.0,
                (Sounds.Ki1, Sounds.Ki1): 0.1,
                (Sounds.Ki1, Sounds.Ko1): 0.1,
                (Sounds.Ko1, Sounds.Ka1): 0.1,
                (Sounds.Ko1, Sounds.Ki1): 0.1,
                (Sounds.Ko1, Sounds.Ko1): 0.1,
            },
            (Sounds.Ki1, Sounds.Ki1): {
                (Sounds.Ka1, Sounds.Ka1): 0.1,
                (Sounds.Ka1, Sounds.Ki1): 0.1,
                (Sounds.Ka1, Sounds.Ko1): 0.1,
                (Sounds.Ki1, Sounds.Ka1): 0.1,
                (Sounds.Ki1, Sounds.Ki1): 0.0,
                (Sounds.Ki1, Sounds.Ko1): 0.1,
                (Sounds.Ko1, Sounds.Ka1): 0.1,
                (Sounds.Ko1, Sounds.Ki1): 0.1,
                (Sounds.Ko1, Sounds.Ko1): 0.1,
            },
            (Sounds.Ki1, Sounds.Ko1): {
                (Sounds.Ka1, Sounds.Ka1): 0.1,
                (Sounds.Ka1, Sounds.Ki1): 0.1,
                (Sounds.Ka1, Sounds.Ko1): 0.1,
                (Sounds.Ki1, Sounds.Ka1): 0.1,
                (Sounds.Ki1, Sounds.Ki1): 0.1,
                (Sounds.Ki1, Sounds.Ko1): 0.0,
                (Sounds.Ko1, Sounds.Ka1): 0.1,
                (Sounds.Ko1, Sounds.Ki1): 0.1,
                (Sounds.Ko1, Sounds.Ko1): 0.1,
            },
            (Sounds.Ko1, Sounds.Ka1): {
                (Sounds.Ka1, Sounds.Ka1): 0.1,
                (Sounds.Ka1, Sounds.Ki1): 0.1,
                (Sounds.Ka1, Sounds.Ko1): 0.1,
                (Sounds.Ki1, Sounds.Ka1): 0.1,
                (Sounds.Ki1, Sounds.Ki1): 0.1,
                (Sounds.Ki1, Sounds.Ko1): 0.1,
                (Sounds.Ko1, Sounds.Ka1): 0.0,
                (Sounds.Ko1, Sounds.Ki1): 0.1,
                (Sounds.Ko1, Sounds.Ko1): 0.1,
            },
            (Sounds.Ko1, Sounds.Ki1): {
                (Sounds.Ka1, Sounds.Ka1): 0.1,
                (Sounds.Ka1, Sounds.Ki1): 0.1,
                (Sounds.Ka1, Sounds.Ko1): 0.1,
                (Sounds.Ki1, Sounds.Ka1): 0.1,
                (Sounds.Ki1, Sounds.Ki1): 0.1,
                (Sounds.Ki1, Sounds.Ko1): 0.1,
                (Sounds.Ko1, Sounds.Ka1): 0.1,
                (Sounds.Ko1, Sounds.Ki1): 0.0,
                (Sounds.Ko1, Sounds.Ko1): 0.1,
            },
            (Sounds.Ko1, Sounds.Ko1): {
                (Sounds.Ka1, Sounds.Ka1): 0.1,
                (Sounds.Ka1, Sounds.Ki1): 0.1,
                (Sounds.Ka1, Sounds.Ko1): 0.1,
                (Sounds.Ki1, Sounds.Ka1): 0.1,
                (Sounds.Ki1, Sounds.Ki1): 0.1,
                (Sounds.Ki1, Sounds.Ko1): 0.1,
                (Sounds.Ko1, Sounds.Ka1): 0.1,
                (Sounds.Ko1, Sounds.Ki1): 0.1,
                (Sounds.Ko1, Sounds.Ko1): 0.0,
            },
        },
        (Sounds.Ka1, Sounds.Ka1),
    )

    short_reverse_chain = MarkovChain({True: {True: 0.0, False: 1.0}, False: {True: 0.1, False: 0.9}}, False)

    short_start_chain = MarkovChain(
        {
            TimeStart.EARLY: {TimeStart.EARLY: 0.2, TimeStart.MIDDLE: 0.4, TimeStart.LATE: 0.4},
            TimeStart.MIDDLE: {TimeStart.EARLY: 0.4, TimeStart.MIDDLE: 0.2, TimeStart.LATE: 0.4},
            TimeStart.LATE: {TimeStart.EARLY: 0.4, TimeStart.MIDDLE: 0.4, TimeStart.LATE: 0.2},
        },
        TimeStart.MIDDLE,
    )

    def play_long_sound(self, start: float, long_filter_type_chain: MarkovChain) -> float:
        max_duration = 0

        long_sounds = self.long_sound_chain.next()
        for long_sound in long_sounds:
            sound_data = all_sound_data[long_sound]
            # print(sound_data.make_fact())
            rate = random_range(0.95, 1.05)
            # rate = random_range(0.2, 0.4)
            max_duration = max(sound_data.get_play_duration(rate), max_duration)
            reverse = self.long_reverse_chain.next()
            sound_start_time, sound_end_time = sound_data.start_time, sound_data.end_time
            if reverse:
                sound_start_time, sound_end_time = sound_end_time, sound_start_time
            filter_type = long_filter_type_chain.next()

            self.play_filtered_sound(
                start_time=start,
                rate=rate,
                sound=long_sound,
                filter_type=filter_type,
                sound_start_time=sound_start_time,
                sound_end_time=sound_end_time,
            )
        return max_duration

    def play_short_sound(self, start: float, short_filter_type_chain: MarkovChain) -> float:
        max_duration = 0
        short_sounds = self.short_sound_chain.next()
        for short_sound in short_sounds:
            sound_data = all_sound_data[short_sound]
            # print(sound_data.make_fact())
            rate = random_range(0.95, 1.05)
            # rate = random_range(0.2, 0.4)
            max_duration = max(sound_data.get_play_duration(rate), max_duration)
            reverse = self.short_reverse_chain.next()
            sound_start_time, sound_end_time = sound_data.start_time, sound_data.end_time
            if reverse:
                sound_start_time, sound_end_time = sound_end_time, sound_start_time
            filter_type = short_filter_type_chain.next()
            self.play_filtered_sound(
                start_time=start,
                rate=rate,
                sound=short_sound,
                filter_type=filter_type,
                sound_start_time=sound_start_time,
                sound_end_time=sound_end_time,
            )
        return max_duration

    def play_sound_gesture(
        self, start: float, long_filter_type_chain: MarkovChain, short_filter_type_chain: MarkovChain
    ):
        long_duration = self.play_long_sound(start, long_filter_type_chain)
        short_start_type = self.short_start_chain.next()
        match short_start_type:
            case TimeStart.EARLY:
                start_delay_range = random_range(0, 0.25)
            case TimeStart.MIDDLE:
                start_delay_range = random_range(0.33, 0.66)
            case TimeStart.LATE:
                start_delay_range = random_range(0.75, 0.99)
        short_start = start + (long_duration * start_delay_range)
        self.play_short_sound(short_start, short_filter_type_chain)

    def handle_sound_types1(self, patch_arguments: PatchArguments):
        match patch_arguments.note:
            case 0:
                self.play_long_sound(patch_arguments.start, self.low_filter_type_chain)
            case 1:
                self.play_long_sound(patch_arguments.start, self.high_filter_type_chain)
            case 2:
                self.play_short_sound(patch_arguments.start, self.low_filter_type_chain)
            case 3:
                self.play_short_sound(patch_arguments.start, self.high_filter_type_chain)
            case 4:
                self.play_sound_gesture(patch_arguments.start, self.low_filter_type_chain, self.low_filter_type_chain)
            case 5:
                self.play_sound_gesture(patch_arguments.start, self.high_filter_type_chain, self.high_filter_type_chain)
            case 6:
                self.play_sound_gesture(patch_arguments.start, self.low_filter_type_chain, self.high_filter_type_chain)
            case 7:
                self.play_sound_gesture(patch_arguments.start, self.high_filter_type_chain, self.low_filter_type_chain)
            case _:
                pass

    def get_grain_duration(self, duration: Duration) -> float:
        match duration:
            case Duration.SHORT:
                return random_range(0.01, 0.03)
            case Duration.MEDIUM:
                return random_range(0.1, 0.3)
            case Duration.LONG:
                return random_range(1.5, 1.9)
            case _:
                return 0.1

    status = widgets.Output()
    display(status)

    def play_grain_buf(
        self,
        start: float,
        sound_data: SoundData,
        note: int,
        rate_note: int,
        grain_pos_reverse: bool,
        grain_duration_type: Duration,
        duration: float,
        volume: float,
    ):
        sound = sound_data.sound
        pan = random_range(-0.99, 0.99)

        spectrum = sound_data.make_undertone_spectrum()
        freq = spectrum[note]
        freq_control = static_control(freq)
        rate = sound_data.make_rate(spectrum[rate_note])

        grain_trigger_bus = impulse_osc(static_control(1.0), freq_control).add_action(AddAction.TAIL_ACTION)
        grain_duration_bus = static_control(self.get_grain_duration(grain_duration_type))
        grain_rate_bus = static_control(rate)

        relative_start, relative_end = sound_data.get_relative_start_end()

        pos_start = relative_start * random_range(1.0, 1.2)
        pos_end = relative_end * random_range(0.8, 0.95)
        if grain_pos_reverse:
            pos_start, pos_end = pos_end, pos_start

        grain_pos_bus = line_control(pos_start, pos_end)

        (
            piece.synth_player.note()
            .mono_grain_buf(sound, grain_trigger_bus, grain_duration_bus, grain_rate_bus, grain_pos_bus)
            .mono_volume(sine_control(0, volume))
            .pan(static_control(pan))
            .play(start_time=start, duration=duration)
        )
        self.status.clear_output()
        with self.status:
            display(f"Rate {rate_note} ({rate}) note {note} (freq)")

    def handle_grain_buf1(self, patch_arguments: PatchArguments):
        sound_data = all_sound_data[Sounds.H1]

        match patch_arguments.octave:
            case 2:
                self._rate_note = patch_arguments.note
            case 3:
                note = patch_arguments.note
                rate_note = self._rate_note

                grain_pos_reverse = False
                grain_duration_type = Duration.SHORT
                duration = random_range(5, 8)
                volume = volume = random_range(0.35, 0.99) * 4.0
                self.play_grain_buf(
                    patch_arguments.start,
                    sound_data,
                    note,
                    rate_note,
                    grain_pos_reverse,
                    grain_duration_type,
                    duration,
                    volume,
                )
            case _:
                pass

    def get_rate_note(self, note_type: NoteType) -> int:
        match note_type:
            case NoteType.LOW:
                return random_int_range(0, 2)
            case NoteType.MIDDLE_LOW:
                return random_int_range(3, 4)
            case NoteType.MIDDLE_HIGH:
                return random_int_range(5, 7)
            case NoteType.HIGH:
                return random_int_range(8, 9)

    def handle_grain_buf2(self, patch_arguments: PatchArguments):
        sound_data = all_sound_data[Sounds.H1]
        match patch_arguments.octave:
            case 2:
                rate_note_type = NoteType.LOW
            case 3:
                rate_note_type = NoteType.MIDDLE_LOW
            case 4:
                rate_note_type = NoteType.MIDDLE_HIGH
            case 5:
                rate_note_type = NoteType.HIGH

        note = patch_arguments.note
        rate_note = self.get_rate_note(rate_note_type)
        grain_pos_reverse = False
        grain_duration_type = Duration.SHORT
        duration = random_range(5, 8)
        volume = volume = random_range(0.35, 0.99) * 4.0
        self.play_grain_buf(
            patch_arguments.start, sound_data, note, rate_note, grain_pos_reverse, grain_duration_type, duration, volume
        )

    # rate note groups
    # 0, 1, 2
    # 3, 4
    # 5, 6. 7
    # 8, 9, 10, 11

    # note groups
    # 2
    # 0 1, 2 3, 4 5, 6 7, 8 9, 10 11
    # 0 1, 1 0
    # 3
    # 0 1 2, 3 4 5, 6 7 8, 9 10 11
    #
    # 4
    # 0 1 2 3, 4 5 6 7, 8 9 10 11

    two_groups = [(0, 1), (2, 3), (4, 5), (6, 7), (8, 9), (10, 11)]
    three_groups = [(0, 1, 2), (3, 4, 5), (6, 7, 8), (9, 10, 11)]
    four_groups = [(0, 1, 2, 3), (4, 5, 6, 7), (8, 9, 10, 11)]

    def get_permutaions(self, size: int) -> list[tuple[int]]:
        t = tuple(range(0, size))
        return list(permutations(t))

    def get_random_permutations(self, t_size: int, nr_of_permutations: int) -> list[tuple[int]]:
        p = self.get_permutaions(t_size)
        return random.choices(p, k=nr_of_permutations)

    rate_note_type_chain1 = MarkovChain(
        {
            NoteType.LOW: {
                NoteType.LOW: 0.0,
                NoteType.MIDDLE_LOW: 0.5,
                NoteType.MIDDLE_HIGH: 0.25,
                NoteType.HIGH: 0.25,
            },
            NoteType.MIDDLE_LOW: {
                NoteType.LOW: 0.1,
                NoteType.MIDDLE_LOW: 0.3,
                NoteType.MIDDLE_HIGH: 0.4,
                NoteType.HIGH: 0.2,
            },
            NoteType.MIDDLE_HIGH: {
                NoteType.LOW: 0.2,
                NoteType.MIDDLE_LOW: 0.4,
                NoteType.MIDDLE_HIGH: 0.1,
                NoteType.HIGH: 0.3,
            },
            NoteType.HIGH: {NoteType.LOW: 0.2, NoteType.MIDDLE_LOW: 0.5, NoteType.MIDDLE_HIGH: 0.3, NoteType.HIGH: 0.0},
        },
        NoteType.MIDDLE_LOW,
    )

    def play_grain_buf_group1(self, start: float, note_groups_size: int):
        match note_groups_size:
            case 2:
                gesture_size = 2
                note_groups = self.two_groups
            case 3:
                gesture_size = 3
                note_groups = self.three_groups
            case 4:
                gesture_size = 4
                note_groups = self.four_groups
            case _:
                pass

        groups = random.sample(note_groups, k=2)
        time = 0

        for group in groups:
            group_orders = self.get_random_permutations(gesture_size, 2)
            for group_order in group_orders:
                for pos in group_order:
                    note = group[pos]
                    rate_note_type = self.rate_note_type_chain1.next()
                    rate_note = self.get_rate_note(rate_note_type)
                    grain_pos_reverse = False
                    grain_duration_type = Duration.SHORT
                    # duration = random_range(5, 8)
                    duration = random_range(8, 13)
                    volume = volume = random_range(0.35, 0.99) * 4.0
                    absolute_time = start + time
                    self.play_grain_buf(
                        absolute_time,
                        sound_data,
                        note,
                        rate_note,
                        grain_pos_reverse,
                        grain_duration_type,
                        duration,
                        volume,
                    )
                    # time += random_range(3, 5)
                    time += random_range(5, 8)
                time = 0

    def handle_grain_buf_group1(self, patch_arguments: PatchArguments):
        match patch_arguments.octave:
            case 2:

                match patch_arguments.note:
                    case 0:
                        self.play_grain_buf_group1(patch_arguments.start, 2)
                    case 1:
                        self.play_grain_buf_group1(patch_arguments.start, 3)
                    case 2:
                        self.play_grain_buf_group1(patch_arguments.start, 4)
                    case _:
                        pass
            case 3:
                match patch_arguments.note:
                    case 0:
                        self.play_long_sound(patch_arguments.start, self.low_filter_type_chain)
                    case 1:
                        self.play_long_sound(patch_arguments.start, self.high_filter_type_chain)
                    case 2:
                        self.play_short_sound(patch_arguments.start, self.low_filter_type_chain)
                    case 3:
                        self.play_short_sound(patch_arguments.start, self.high_filter_type_chain)
                    case 4:
                        self.play_sound_gesture(
                            patch_arguments.start, self.low_filter_type_chain, self.low_filter_type_chain
                        )
                    case 5:
                        self.play_sound_gesture(
                            patch_arguments.start, self.high_filter_type_chain, self.high_filter_type_chain
                        )
                    case 6:
                        self.play_sound_gesture(
                            patch_arguments.start, self.low_filter_type_chain, self.high_filter_type_chain
                        )
                    case 7:
                        self.play_sound_gesture(
                            patch_arguments.start, self.high_filter_type_chain, self.low_filter_type_chain
                        )
                    case _:
                        pass
            case _:
                pass

    def _get_variable_start_end(self, the_arg: float | tuple[float, float] | None) -> tuple[float | None, float | None]:
        match the_arg:
            case (start, end):
                start = start
                end = end
            case float(val) | int(val):
                start = val
                end = val
            case None:
                start = None
                end = None

        return (start, end)

    def make_grain_trigger_bus(
        self, impulse_freq: float | tuple[float, float] | None, dust_freq: float | tuple[float, float] | None = None
    ) -> AudioInstrument:
        impulse_start, impulse_end = self._get_variable_start_end(impulse_freq)
        if impulse_start:
            impulse_trigger = impulse_osc(
                amp_bus=static_control(1.0), freq_bus=line_control(impulse_start, impulse_end)
            ).add_action(AddAction.TAIL_ACTION)

        dust_start, dust_end = self._get_variable_start_end(dust_freq)
        if dust_start:
            dust_trigger = dust_osc(
                amp_bus=static_control(1.0), freq_bus=line_control(dust_start, dust_end)
            ).add_action(AddAction.TAIL_ACTION)

        match (impulse_trigger, dust_trigger):
            case (imp, dust):
                return signal_sum(imp, dust).add_action(AddAction.TAIL_ACTION)
            case (imp, None) if imp is not None:
                return imp
            case (None, dust) if dust is not None:
                return dust

    def make_grain_duration_bus(
        self,
        grain_duration: float | tuple[float, float],
        grain_duration_noise: float | tuple[float, float] | None = None,
        lf_noise_rate: float = 500,
    ) -> AudioInstrument:
        grain_duration_start, grain_duration_end = self._get_variable_start_end(grain_duration)
        grain_duration_line = line_control(grain_duration_start, grain_duration_end).add_action(AddAction.TAIL_ACTION)
        match grain_duration_noise:
            case (noise_lower, noise_upper):
                grain_duration_lf_noise = lf_noise_osc(
                    amp_bus=static_control(1.0),
                    freq_bus=static_control(lf_noise_rate),
                    low_value=noise_lower,
                    high_value=noise_upper,
                )
            case float(noise_val) | int(noise_val):
                grain_duration_lf_noise = lf_noise_osc(
                    amp_bus=static_control(1.0),
                    freq_bus=static_control(lf_noise_rate),
                    low_value=noise_val if noise_val < 1.0 else 1.0,
                    high_value=noise_val if noise_val > 1.0 else 1.0,
                )
            case None:
                grain_duration_lf_noise = None

        if grain_duration_lf_noise is not None:
            return signal_multiply(grain_duration_line, grain_duration_lf_noise).add_action(AddAction.TAIL_ACTION)
        else:
            return grain_duration_line

    def make_grain_rate_bus(
        self,
        grain_rate: float | tuple[float, float],
        grain_rate_noise: float | tuple[float, float] | None = None,
        lf_noise_rate: float = 500,
    ) -> AudioInstrument:
        grain_rate_start, grain_rate_end = self._get_variable_start_end(grain_rate)
        grain_rate_line = line_control(grain_rate_start, grain_rate_end).add_action(AddAction.TAIL_ACTION)
        match grain_rate_noise:
            case (noise_lower, noise_upper):
                grain_rate_lf_noise = lf_noise_osc(
                    amp_bus=static_control(1.0),
                    freq_bus=static_control(lf_noise_rate),
                    low_value=noise_lower,
                    high_value=noise_upper,
                )
            case float(noise_val) | int(noise_val):
                grain_rate_lf_noise = lf_noise_osc(
                    amp_bus=static_control(1.0),
                    freq_bus=static_control(lf_noise_rate),
                    low_value=noise_val if noise_val < 1.0 else 1.0,
                    high_value=noise_val if noise_val > 1.0 else 1.0,
                )
            case None:
                grain_rate_lf_noise = None

        if grain_rate_lf_noise is not None:
            return signal_multiply(grain_rate_line, grain_rate_lf_noise).add_action(AddAction.TAIL_ACTION)
        else:
            return grain_rate_line

    def make_grain_pos_bus(
        self,
        sound_data: SoundData,
        reversed: bool,
        grain_pos_noise: float | tuple[float, float] | None = None,
        lf_noise_rate: float = 500,
    ) -> AudioInstrument:
        relative_start, relative_end = sound_data.get_relative_start_end()

        if reversed:
            pos_end = relative_start
            pos_start = relative_end
        else:
            pos_start = relative_start
            pos_end = relative_end

        line_pos = line_control(pos_start, pos_end).add_action(AddAction.TAIL_ACTION)

        match grain_pos_noise:
            case (noise_lower, noise_upper):
                grain_pos_lf_noise = lf_noise_osc(
                    amp_bus=static_control(1.0),
                    freq_bus=static_control(lf_noise_rate),
                    low_value=noise_lower,
                    high_value=noise_upper,
                )
            case float(noise_val) | int(noise_val):
                grain_pos_lf_noise = lf_noise_osc(
                    amp_bus=static_control(1.0),
                    freq_bus=static_control(lf_noise_rate),
                    low_value=noise_val * -1.0,
                    high_value=noise_val,
                )
            case None:
                grain_pos_lf_noise = None

        if grain_pos_lf_noise is not None:
            return signal_sum(line_pos, grain_pos_lf_noise).add_action(AddAction.TAIL_ACTION)
        else:
            return line_pos

    def play_sound_with_grainbuf(
        self,
        start: float,
        sound_data: SoundData,
        grain_pos_bus: AudioInstrument,        
        pan_position: AudioInstrument,
        grain_trigger_bus: AudioInstrument,
        grain_duration_bus: AudioInstrument,
        grain_rate_bus: AudioInstrument,
        volume: float,
        play_rate: float,        
    ):
        sound = sound_data.sound
        
        duration = sound_data.get_play_duration(play_rate)
        (
            piece.synth_player.note()
            .mono_grain_buf(sound, grain_trigger_bus, grain_duration_bus, grain_rate_bus, grain_pos_bus)
            .mono_volume(sine_control(0, volume))
            .pan(pan_position)
            .play(start_time=start, duration=duration)
        )

    def grainbuf_play_sound1(self, patch_arguments: PatchArguments):
        match patch_arguments.octave:
            case 2:

                match patch_arguments.note:
                    case 0:
                        self.play_grain_buf_group1(patch_arguments.start, 2)
                    case 1:
                        self.play_grain_buf_group1(patch_arguments.start, 3)
                    case 2:
                        self.play_grain_buf_group1(patch_arguments.start, 4)
                    case _:
                        pass
            case 3:
                sound_data = all_sound_data[Sounds.H1]
                undertone_spectrum = sound_data.make_undertone_spectrum()
                spectrum = sound_data.make_spectrum()
                pan_position = line_control(random_range(-0.99, 0.99), random_range(-0.99, 0.99))
                # trigger_freq = 40
                trigger_freq = random_range(0.3, 10)
                # trigger_freq = random_range(0.3, 0.5)
                # trigger_freq = 2
                grain_trigger_bus = self.make_grain_trigger_bus(impulse_freq=trigger_freq, dust_freq=trigger_freq)
                grain_duration = (1 / trigger_freq) * random_range(0.50, 0.75)
                grain_duration_bus = self.make_grain_duration_bus(
                    grain_duration=grain_duration, grain_duration_noise=(0.85, 1.15)
                )
                grain_rate = sound_data.make_rate(undertone_spectrum[patch_arguments.note])
                grain_rate2 = sound_data.make_rate(undertone_spectrum[patch_arguments.note + 2])
                grain_rate_bus = self.make_grain_rate_bus(
                    grain_rate=(grain_rate, grain_rate2), grain_rate_noise=(0.85, 1.15)
                )
                volume = 1.0
                play_rate = random_range(5, 8)
                reversed = random.choice([True, False])
                grain_pos_bus = self.make_grain_pos_bus(sound_data=sound_data, reversed=reversed, grain_pos_noise=0.01)
                self.play_sound_with_grainbuf(
                    patch_arguments.start,
                    sound_data=sound_data,
                    grain_pos_bus=grain_pos_bus,
                    pan_position=pan_position,
                    grain_trigger_bus=grain_trigger_bus,
                    grain_duration_bus=grain_duration_bus,
                    grain_rate_bus=grain_rate_bus,
                    volume=volume,
                    play_rate=play_rate,                    
                )
            case 4:
                sound_data = all_sound_data[Sounds.H1]
                undertone_spectrum = sound_data.make_undertone_spectrum()
                spectrum = sound_data.make_spectrum()
                pan_position = line_control(random_range(-0.99, 0.99), random_range(-0.99, 0.99))
                trigger_freq = 40
                grain_trigger_bus = self.make_grain_trigger_bus(impulse_freq=40, dust_freq=40)
                grain_duration = 0.3
                grain_duration_bus = self.make_grain_duration_bus(
                    grain_duration=grain_duration, grain_duration_noise=None
                )
                grain_rate = sound_data.make_rate(spectrum[patch_arguments.note])
                grain_rate2 = sound_data.make_rate(spectrum[patch_arguments.note + 2])
                grain_rate_bus = self.make_grain_rate_bus(
                    grain_rate=(grain_rate, grain_rate2), grain_rate_noise=(0.85, 1.15)
                )
                volume = 1.0
                play_rate = random_range(5, 8)

                reversed = random.choice([True, False])
                grain_pos_bus = self.make_grain_pos_bus(sound_data=sound_data, reversed=reversed, grain_pos_noise=0.01)
                self.play_sound_with_grainbuf(
                    patch_arguments.start,
                    sound_data=sound_data,
                    grain_pos_bus=grain_pos_bus,
                    pan_position=pan_position,
                    grain_trigger_bus=grain_trigger_bus,
                    grain_duration_bus=grain_duration_bus,
                    grain_rate_bus=grain_rate_bus,
                    volume=volume,
                    play_rate=play_rate                    
                )
            case _:
                pass


    def compare_grainbuf_func(self, patch_arguments: PatchArguments):
        sound_data = all_sound_data[Sounds.H1]
        match patch_arguments.octave:
            case 2:                
                rate_note_type = NoteType.LOW
                #rate_note_type = NoteType.MIDDLE_LOW        
                #rate_note_type = NoteType.MIDDLE_HIGH        
                #rate_note_type = NoteType.HIGH
                note = patch_arguments.note
                rate_note = self.get_rate_note(rate_note_type)
                grain_pos_reverse = False
                grain_duration_type = Duration.SHORT
                duration = random_range(5, 8)
                volume = volume = random_range(0.35, 0.99) * 4.0
                self.play_grain_buf(
                    patch_arguments.start, sound_data, note, rate_note, grain_pos_reverse, grain_duration_type, duration, volume
                )
            case 3:
                
                relative_start, relative_end = sound_data.get_relative_start_end()
                pos_start = relative_start * random_range(1.0, 1.2)
                pos_end = relative_end * random_range(0.8, 0.95)                
                grain_pos_bus = line_control(pos_start, pos_end)
                
                pan_position = static_control(random_range(-0.99, 0.99))

                spectrum = sound_data.make_undertone_spectrum()
                note = patch_arguments.note
                freq = spectrum[note]
                freq_control = static_control(freq)        
                grain_trigger_bus = impulse_osc(static_control(1.0), freq_control).add_action(AddAction.TAIL_ACTION)

                grain_duration_type = Duration.SHORT
                grain_duration_bus = static_control(self.get_grain_duration(grain_duration_type))

                rate_note_type = NoteType.LOW
                rate_note = self.get_rate_note(rate_note_type)
                rate = sound_data.make_rate(spectrum[rate_note])
                grain_rate_bus = static_control(rate)

                volume = volume = random_range(0.35, 0.99) * 4.0

                play_rate = random_range(5, 8)

                self.play_sound_with_grainbuf(
                    patch_arguments.start,
                    sound_data=sound_data,
                    grain_pos_bus=grain_pos_bus,
                    pan_position=pan_position,
                    grain_trigger_bus=grain_trigger_bus,
                    grain_duration_bus=grain_duration_bus,
                    grain_rate_bus=grain_rate_bus,
                    volume=volume,
                    play_rate=play_rate                    
                )
            case _:
                return

    def handle_note(self, patch_arguments: PatchArguments):
        self.compare_grainbuf_func(patch_arguments)


my_handler2 = MyHandler2(piece.supercollider_client)
piece.receiver.set_note_handler(my_handler2)

Output()

In [ ]:
piece.stop()